# ETL Pipeline: Knapsack Problem Instance Processing
**Author**  
Paola A. Castillo-Gutierrez   
A00843399   
**MCC Thesis**  
Exploring a Continual Learning Approach for Algorithm Selection under
Incremental Distribution Shift in the 0/1 Knapsack Problem


## Overview

This notebook implements an **Extract-Transform-Load (ETL)** pipeline for processing synthetic 0/1 Knapsack Problem (KP) instances generated via the evolutionary-based approach described in Plata-González et al. (2019).

The pipeline performs three stages:

1. **Extract**: Read raw `.kp` instance files from organized subdirectories, parse each file's structure (header metadata + item rows), and assign labels (`dominant_heuristic`, `instance_type`) derived from the filename convention.
2. **Transform**: Compute instance-level normalized features from parsed instances and assemble two DataFrames (item-level and feature-level).
3. **Load**: Produce two consolidated CSV outputs (a unified item-level dataset and an instance-level feature dataset).

## Source File Format (`.kp`)

Each `.kp` file encodes a single 0/1 KP instance with the following structure:

| Row | Column 1 | Column 2 |
|-----|----------|----------|
| 1 (header) | `n` (number of items) | `C` (knapsack capacity) |
| 2 to `n+1` | `w_j` (weight of item `j`) | `p_j` (profit of item `j`) |

The files are organized in subdirectories by experimental set (e.g., `Set-25-32`, `Set-200-256`), where the naming convention is `Set-{n_items}-{capacity}`. 

## Instance Features
### Plata-González et al. (2019) features

Seven normalized features characterize each instance, all bounded to $[0, 1]$ and rounded to 3 decimal places:

| Feature | Symbol | Column | Formula |
|---------|--------|--------|---------|
| Mean weight (normalized) | $\bar{w}$ | `w_mean` | $\text{mean}(w) \;/\; \max(w)$ |
| Median weight (normalized) | $\tilde{w}$ | `w_median` | $\text{median}(w) \;/\; \max(w)$ |
| Std. dev. of weights (normalized) | $\sigma_w$ | `w_std` | $\text{std}(w, \text{ddof}=1) \;/\; \max(w)$ |
| Mean profit (normalized) | $\bar{p}$ | `p_mean` | $\text{mean}(p) \;/\; \max(p)$ |
| Median profit (normalized) | $\tilde{p}$ | `p_median` | $\text{median}(p) \;/\; \max(p)$ |
| Std. dev. of profits (normalized) | $\sigma_p$ | `p_std` | $\text{std}(p, \text{ddof}=1) \;/\; \max(p)$ |
| Weight-profit correlation (shifted) | $r$ | `wp_corr` | $\text{corr}(w, p) \;/\; 2 + 0.5$ |

The standard deviation uses the sample estimator (`ddof=1`), consistent with the paper's worked example (Section 2.2, p. 12713).

The correlation transformation maps Pearson's $r \in [-1, 1]$ to $[0, 1]$.

Additionally, the raw Pearson correlation is stored in `wp_corr_raw` ($r \in [-1, 1]$) to preserve the unshifted value, which matches the paper's worked example directly (see Section 5 below for reconciliation).

### Additional features (not in Plata-González et al., 2019)

| Feature | Column | Formula | Range |
|---------|--------|---------|-------|
| Capacity ratio | `capacity_ratio` | $C \;/\; \sum_{j=1}^{n} w_j$ | $[0, \infty)$ |

This ratio measures the relative tightness of the knapsack constraint. A value close to 0 indicates a highly constrained instance; a value $\geq 1$ means the capacity is non-binding (all items can be packed). This is a standard descriptor in the KP literature for characterizing instance difficulty (Pisinger, 2005).

Four raw range descriptors are also included per instance (not normalized): `weight_min`, `weight_max`, `profit_min`, `profit_max`.

### References (APA 7)

1. Plata-González, L. F., Amaya, I., Ortiz-Bayliss, J. C., Conant-Pablos, S. E., Terashima-Marín, H., & Coello Coello, C. A. (2019). Evolutionary-based tailoring of synthetic instances for the Knapsack problem. *Soft Computing*, *23*, 12711–12728.https://doi.org/10.1007/s00500-019-03822-w

2. Pisinger, D. (2005). Where are the hard knapsack problems? *Computers & Operations Research*, *32*(9), 2271–2284.https://doi.org/10.1016/j.cor.2004.03.002

In [1]:
import os # Fallback para posibles operaciones a nivel del sistema operativo
import pandas as pd # Crea y manipula DataFrames, exporta dataframes a CSV 
import numpy as np # Realiza cálculos numéricos en arrays
from pathlib import Path # Manejo orientado a objetos de rutas de archivos
from dataclasses import dataclass # Genera automáticamente métodos como __init__ y __repr__ para la clase

In [2]:
# Importa el módulo drive de Google Colab y lo monta en /content/drive
# Esto permite acceder a los archivos almacenados en Google Drive desde el notebook
# from google.colab import drive
# drive.mount('/content/drive')

## Configuration

In [3]:
# Directorio base (google drive) con las instancias KP
# BASE_DIR = Path('/content/drive/MyDrive/KP Instances 2026')

# OUTPUT_DIR = Path('outputs')
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
# BASE_DIR apunta a la subcarpeta instances relativa al notebook
BASE_DIR = Path('KP instances 2026')

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Domain Model

`KPInstance` is a dataclass that encapsulates the raw data and metadata of a single 0/1 KP instance. The class factory `from_file` handles:

- Parsing the two-column text format (header row + item rows).
- Validating that the declared number of items matches the actual row count.
- Extracting the dominant heuristic and instance difficulty type from the filename convention (e.g., `DEF_EASY_100_000.kp` $\rightarrow$ heuristic=`DEF`, type=`easy`).

In [5]:
@dataclass # Define KPInstance como una clase de datos para almacenar la información de cada instancia KP
class KPInstance:
    filename: str # String 
    set_name: str # String 
    n_items: int  # Integer 
    capacity: int # Integer 
    weights: np.ndarray #  Array con memoria contigua, eficiente para cálculos numéricos, mismo tipo de dato
    profits: np.ndarray #  Array con memoria contigua, eficiente para cálculos numéricos, mismo tipo de dato
    dominant_heuristic: str # String 
    instance_type: str # String 

    @staticmethod # Método de fábrica para crear una instancia KP a partir de un archivo
    def from_file(filepath: Path, set_name: str) -> 'KPInstance':
        # Lee el archivo y tokeniza cada línea, eliminando comas
        lines = filepath.read_text().strip().splitlines()
        tokens = [[t.strip(',') for t in line.split()] for line in lines if line.strip()]

        # Primera fila: encabezado con n_items y capacity
        n_items = int(tokens[0][0])
        capacity = int(tokens[0][1])

        # Filas restantes: pares (weight, profit) de cada item
        items = np.array(tokens[1:], dtype=float)
        weights = items[:, 0]
        profits = items[:, 1]

        # Valida que la cantidad de items coincida con la declaración del encabezado
        if len(weights) != n_items:
            raise ValueError(
                f"{filepath.name}: header declares {n_items} items "
                f"but file contains {len(weights)} item rows"
            )

        # Extrae etiquetas (heurística y tipo) de la convención de nombres de archivo
        dominant_heuristic, instance_type = KPInstance._parse_label(filepath.stem)

        return KPInstance(
            filename=filepath.name,
            set_name=set_name,
            n_items=n_items,
            capacity=capacity,
            weights=weights,
            profits=profits,
            dominant_heuristic=dominant_heuristic,
            instance_type=instance_type,
        )

    @staticmethod
    def _parse_label(stem: str) -> tuple:
        # Busca la etiqueta _EASY o _HARD en el nombre del archivo (sin distinguir mayúsculas/minúsculas)
        # Esto no es necesario para los datos proporcionados para la tesis, pero hace el código más robusto ante posibles variaciones en el formato de los nombres de archivo
        stem_upper = stem.upper()
        if '_EASY' in stem_upper:
            tag = '_EASY'
        elif '_HARD' in stem_upper:
            tag = '_HARD'
        else:
            return ('unknown', 'unknown')
        
        # Extrae nombre de heurística (antes de etiqueta, preserva caso) y tipo de instancia (easy/hard)
        idx = stem_upper.index(tag)
        heuristic = stem[:idx]
        instance_type = tag.strip('_').lower()
        return (heuristic, instance_type)

## 1.1. Feature Extractor

`KPFeatureExtractor` computes the seven normalized instance features defined in Plata-González et al. (2019), plus `capacity_ratio` (Pisinger, 2005) and four raw range descriptors.

Logical flow:
1. Cast weight/profit vectors to float and extract their maxima.
2. Guard against degenerate instances where $\max(w) = 0$ or $\max(p) = 0$ (returns all NaN).
3. Compute the six normalized location/dispersion features by dividing by the respective maximum.
4. Compute both the raw Pearson correlation (`wp_corr_raw`) and its $[0,1]$-shifted version (`wp_corr`) in a single call.
5. Compute `capacity_ratio` as $C / \sum w_j$.
6. Validate that all features expected to lie in $[0, 1]$ actually do so.

In [6]:
class KPFeatureExtractor:
    # Clase responsable de extraer y calcular 13 características normalizadas de cada instancia KP
    # Estas características se utilizan como entrada para modelos de machine learning
    
    # Número de decimales para redondear todas las características normalizadas
    # Se utiliza para mantener consistencia y precisión en los cálculos
    DECIMALS = 3

    @classmethod
    def compute(cls, instance: KPInstance) -> dict:
        # MÉTODO PRINCIPAL: Calcula todas las 13 características de una instancia KP
        # Entrada: una instancia KP con arrays de weights y profits
        # Salida: diccionario con 13 características calculadas y normalizadas
        # Total de características: 6 (dispersión) + 2 (correlación) + 4 (rangos) + 1 (ratio capacidad) = 13
        
        # Paso 1: Convierte weights y profits a tipo float para operaciones numéricas
        w = instance.weights.astype(float)
        p = instance.profits.astype(float)
        # Obtiene los valores máximos para normalización posterior
        max_w = w.max()
        max_p = p.max()

        # Paso 2: Protección contra instancias degeneradas
        # Si todos los pesos o todas las ganancias son cero, no se puede normalizar
        # En este caso, retorna un diccionario completo con valores NaN
        if max_w == 0 or max_p == 0:
            return cls._nan_features()

        # Paso 3: Calcula la correlación de Pearson en ambas formas (bruta y desplazada)
        # Esta correlación mide la relación lineal entre weights y profits
        # Nota: Se calcula UNA SOLA VEZ, sin redundancias
        corr_raw, corr_shifted = cls._shifted_correlation(w, p)

        # Paso 4: Construye diccionario con las 13 características calculadas
        # Las características están agrupadas en 4 grupos según su naturaleza y rango
        features = {
            # GRUPO 1: Características de localización y dispersión de weights (3 características)
            # Normalizadas dividiendo por el máximo peso, resultado en rango [0, 1]
            # Se redondean a DECIMALS decimales para consistencia
            'w_mean':   round(w.mean() / max_w, cls.DECIMALS),          # Media de pesos normalizada
            'w_median': round(float(np.median(w)) / max_w, cls.DECIMALS),  # Mediana de pesos normalizada
            'w_std':    round(w.std(ddof=1) / max_w, cls.DECIMALS),    # Desviación estándar de pesos (ddof=1 para muestra)

            # GRUPO 2: Características de localización y dispersión de profits (3 características)
            # Normalizadas dividiendo por la máxima ganancia, resultado en rango [0, 1]
            # Se redondean a DECIMALS decimales para consistencia
            'p_mean':   round(p.mean() / max_p, cls.DECIMALS),         # Media de ganancias normalizada
            'p_median': round(float(np.median(p)) / max_p, cls.DECIMALS), # Mediana de ganancias normalizada
            'p_std':    round(p.std(ddof=1) / max_p, cls.DECIMALS),   # Desviación estándar de ganancias (ddof=1 para muestra)

            # GRUPO 3: Características de correlación entre weights y profits (2 características)
            # Incluye dos versiones de la misma correlación:
            # - Versión bruta en rango [-1, 1] (coincide con el ejemplo del artículo)
            # - Versión normalizada en rango [0, 1] (para consistencia con otras características)
            'wp_corr_raw': corr_raw,        # Correlación de Pearson bruta en rango [-1, 1]
            'wp_corr':  corr_shifted,       # Correlación de Pearson normalizada en rango [0, 1] (r/2 + 0.5)

            # GRUPO 4: Descriptores de rango SIN normalizar (4 características)
            # Se mantienen los valores originales en escala absoluta para referencia
            # NO están normalizados a [0, 1], mantienen sus valores absolutos originales
            'weight_min': int(w.min()),     # Peso mínimo (valor absoluto, entero)
            'weight_max': int(max_w),       # Peso máximo (valor absoluto, entero)
            'profit_min': int(p.min()),     # Ganancia mínima (valor absoluto, entero)
            'profit_max': int(max_p),       # Ganancia máxima (valor absoluto, entero)

            # GRUPO 5: Medida de restricción de la mochila - capacidad relativa (1 característica)
            # Ratio = capacidad / suma de todos los pesos
            # Valores cercanos a 0: mochila muy ajustada (restricción fuerte)
            # Valores >= 1: mochila no restrictiva (caben todos los items)
            # No se normaliza a [0,1]
            'capacity_ratio': round(instance.capacity / w.sum(), cls.DECIMALS),
        }

        # Paso 5: Valida que todas las características normalizadas estén en rango [0, 1]
        # Si alguna característica está fuera de rango, imprime una advertencia para diagnóstico
        cls._validate_bounds(features, instance.filename)
        
        # Retorna el diccionario completo con las 13 características
        return features

    @classmethod
    def _shifted_correlation(cls, w: np.ndarray, p: np.ndarray) -> tuple:
        # MÉTODO AUXILIAR: Calcula correlación de Pearson en dos formatos
        # Entrada: arrays de weights y profits
        # Salida: tupla (correlación_bruta, correlación_normalizada)
        # NOTA IMPORTANTE: La correlación se calcula UNA SOLA VEZ 
        
        # Verifica si alguno de los vectores es constante (todos los valores iguales)
        # Si es constante, la desviación estándar es 0 y la correlación no está definida
        if w.std() == 0 or p.std() == 0:
            # Retorna valores neutrales (punto medio de cada rango):
            # - 0.0 para correlación bruta (punto medio de [-1, 1])
            # - 0.5 para correlación normalizada (punto medio de [0, 1])
            return (0.0, 0.5)
        
        # Calcula correlación de Pearson bruta UNA SOLA VEZ usando NumPy
        # np.corrcoef retorna una matriz 2x2:
        # - [0,0] y [1,1] son 1.0 (auto-correlación)
        # - [0,1] y [1,0] son la correlación cruzada que necesitamos
        r = float(np.corrcoef(w, p)[0, 1])
        
        # Retorna DOS VERSIONES del mismo valor r calculado (sin recalcular):
        # 1. r en [-1, 1] redondeada a DECIMALS decimales (correlación bruta)
        # 2. r/2 + 0.5 en [0, 1] redondeada a DECIMALS decimales (correlación normalizada)
        # La transformación r/2 + 0.5 mapea linealmente [-1, 1] a [0, 1]:
        # - r = -1 → -1/2 + 0.5 = 0.0 (anticorrelación perfecta → mínimo)
        # - r = 0 → 0/2 + 0.5 = 0.5 (sin correlación → medio)
        # - r = 1 → 1/2 + 0.5 = 1.0 (correlación perfecta → máximo)
        return (round(r, cls.DECIMALS), round(r / 2 + 0.5, cls.DECIMALS))

    @staticmethod
    def _nan_features() -> dict:
        # MÉTODO AUXILIAR: Retorna diccionario con todas las 13 características como NaN
        # Se utiliza cuando la instancia es degenerada (max_w=0 o max_p=0)
        # Esto permite mantener la estructura consistente incluso en casos problemáticos
        
        # Lista de 9 características normalizadas (rango [0, 1] bajo condiciones normales)
        # Incluye todas las características que se espera estén en [0,1]
        norm_keys = [
            'w_mean', 'w_median', 'w_std',      # 3 de dispersión de weights
            'p_mean', 'p_median', 'p_std',      # 3 de dispersión de profits
            'wp_corr_raw', 'wp_corr',           # 2 de correlación (ambas versiones)
            'capacity_ratio',                    # 1 de restricción de capacidad
        ]
        
        # Lista de 4 características sin normalizar (valores absolutos, enteros)
        raw_keys = [
            'weight_min', 'weight_max',         # 2 descriptores de rango de pesos
            'profit_min', 'profit_max',         # 2 descriptores de rango de ganancias
        ]
        
        # Crea diccionario con todas las características normalizadas asignadas a NaN
        result = {k: np.nan for k in norm_keys}
        
        # Añade las características sin normalizar también asignadas a NaN
        result.update({k: np.nan for k in raw_keys})
        
        # Retorna diccionario completo de 13 características, todas con valor NaN
        # Esto asegura consistencia de estructura incluso en instancias degeneradas
        return result

    @staticmethod
    def _validate_bounds(features: dict, filename: str):
        # MÉTODO AUXILIAR: Valida que características normalizadas estén en rango [0, 1]
        # Se ejecuta después de calcular todas las características
        # Sirve como control de calidad para detectar problemas en los cálculos
        
        # Lista de 7 características que DEBEN estar en rango [0, 1]
        # Se excluyen:
        # - wp_corr_raw que está en [-1, 1] (rango diferente, no se valida aquí)
        # - capacity_ratio que puede estar en [0, infinito) (sin límite superior)
        normalized_keys = [
            'w_mean', 'w_median', 'w_std',      # 3 de dispersión de weights
            'p_mean', 'p_median', 'p_std',      # 3 de dispersión de profits
            'wp_corr',                          # 1 de correlación (versión normalizada)
        ]
        
        # Itera sobre cada característica que debe estar en [0, 1]
        for key in normalized_keys:
            val = features[key]
            # Verifica si el valor está fuera del rango válido [0.0, 1.0]
            # Nota: np.isnan(val) retorna True si val es NaN, por lo que ignora NaN naturalmente
            # La condición "not (0.0 <= val <= 1.0)" es False para NaN, así que se ignoran
            if not np.isnan(val) and not (0.0 <= val <= 1.0):
                # Si está fuera de rango, imprime una advertencia con detalles
                # Esto ayuda a identificar y diagnosticar problemas en la normalización
                # Formato: nombre_archivo -> característica=valor fuera de [0, 1]
                print(f"  Advertencia: {filename} -> {key}={val} está fuera de [0, 1]")

## 2. ETL Pipeline

`KPPipeline` orchestrates the three ETL stages:

- **`extract()`**: Iterates over subdirectories, reads each `.kp` file via `KPInstance.from_file`, and collects the parsed instances. Errors are logged without halting execution.
- **`transform()`**: For each parsed instance, (a) expands item-level rows (one row per weight-profit pair) and (b) computes the feature vector via `KPFeatureExtractor.compute`. Produces two DataFrames with enforced column ordering.
- **`load()`**: Writes both DataFrames to CSV in the specified output directory.
- **`run()`**: Convenience method that chains extract $\rightarrow$ transform $\rightarrow$ load.

In [7]:
class KPPipeline:
    # Conjunto de extensiones de archivo válidas que se aceptan como instancias KP
    # (.kp es el formato principal, .txt y .csv son alternativas)
    VALID_EXTENSIONS = {'.kp', '.txt', '.csv'}

    # Ordenamiento canónico de columnas para el DataFrame de características (nivel de instancia)
    # Define el orden exacto en que aparecerán las columnas en el archivo CSV de salida
    FEATURE_COLUMN_ORDER = [
        'filename', 'set', 'dominant_heuristic', 'instance_type',
        'n_items', 'capacity',
        'weight_min', 'weight_max', 'profit_min', 'profit_max',
        'w_mean', 'w_median', 'w_std',
        'p_mean', 'p_median', 'p_std',
        'wp_corr_raw', 'wp_corr',
        'capacity_ratio',
    ]

    # Ordenamiento canónico de columnas para el DataFrame de elementos (nivel de item)
    # Define el orden exacto en que aparecerán las columnas en el archivo CSV de elementos
    ITEM_COLUMN_ORDER = [
        'filename', 'set', 'dominant_heuristic', 'instance_type',
        'weight', 'profit',
    ]

    def __init__(self, base_dir: Path):
        # Inicializa el pipeline con el directorio base que contiene las instancias
        # base_dir: ruta raíz que contiene subcarpetas de conjuntos experimentales
        self.base_dir = base_dir
        # Lista para almacenar todas las instancias KP parseadas exitosamente
        self.instances: list[KPInstance] = []
        # Lista para almacenar mensajes de error durante la extracción (sin detener ejecución)
        self.errors: list[str] = []

    def extract(self) -> 'KPPipeline':
        # ETAPA 1 DEL PIPELINE: Descubre y lee instancias KP desde archivos
        # Busca todas las subcarpetas en el directorio base (cada una representa un conjunto experimental)
        subfolders = sorted(
            [d for d in self.base_dir.iterdir() if d.is_dir()]
        )
        print(f"Found {len(subfolders)} subfolders: {[d.name for d in subfolders]}")

        # Itera sobre cada subcarpeta (conjunto experimental)
        for subfolder in subfolders:
            # Dentro de cada subcarpeta, filtra solo archivos con extensiones válidas
            # y ordena alfabéticamente para reproducibilidad
            files = sorted(
                [f for f in subfolder.iterdir()
                    if f.is_file() and f.suffix in self.VALID_EXTENSIONS]
            )
            count = 0
            # Lee cada archivo individualmente
            for filepath in files:
                try:
                    # Parsea el archivo usando KPInstance.from_file()
                    # Este método valida la estructura del archivo y extrae metadatos del nombre
                    inst = KPInstance.from_file(filepath, subfolder.name)
                    self.instances.append(inst)
                    count += 1
                except Exception as e:
                    # Si ocurre un error, lo registra pero continúa procesando otros archivos
                    # Esto permite robustez ante archivos mal formados
                    self.errors.append(f"{filepath}: {e}")

            print(f"  {subfolder.name}: {count} instances loaded")

        # Imprime resumen de la extracción
        print(f"\nTotal: {len(self.instances)} instances, {len(self.errors)} errors")
        if self.errors:
            # Muestra hasta los primeros 10 errores para diagnóstico
            for err in self.errors[:10]:
                print(f"  ERROR: {err}")
        # Retorna self para permitir encadenamiento de métodos (patrón builder)
        return self

    def transform(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        # ETAPA 2 DEL PIPELINE: Transforma instancias en DataFrames estructurados
        # Crea dos niveles de granularidad:
        # 1. Nivel de elemento: una fila por cada (weight, profit) pair en cada instancia
        # 2. Nivel de característica: una fila por instancia con características calculadas
        
        # Listas temporales para acumular filas antes de convertir a DataFrames
        item_rows = []      # Filas para DataFrame de elementos
        feature_rows = []   # Filas para DataFrame de características

        # Procesa cada instancia de KP extraída
        for inst in self.instances:
            # PARTE 1: Expande filas a nivel de elemento
            # Para cada par (weight, profit) en esta instancia, crea una fila
            for w, p in zip(inst.weights, inst.profits):
                item_rows.append({
                    'filename': inst.filename,
                    'set': inst.set_name,
                    'dominant_heuristic': inst.dominant_heuristic,
                    'instance_type': inst.instance_type,
                    'weight': int(w),
                    'profit': int(p),
                })

            # PARTE 2: Calcula y agrega características a nivel de instancia
            # Usa KPFeatureExtractor para computar las 18 características normalizadas
            features = KPFeatureExtractor.compute(inst)
            
            # Adjunta metadatos de la instancia al diccionario de características
            # Esto vincula cada característica con su instancia de origen
            features.update({
                'filename': inst.filename,
                'set': inst.set_name,
                'dominant_heuristic': inst.dominant_heuristic,
                'instance_type': inst.instance_type,
                'n_items': inst.n_items,
                'capacity': inst.capacity,
            })
            feature_rows.append(features)

        # Construye los DataFrames a partir de las listas de diccionarios
        # Aplica el ordenamiento canónico de columnas especificado en las constantes
        df_items = pd.DataFrame(item_rows)[self.ITEM_COLUMN_ORDER]
        df_features = pd.DataFrame(feature_rows)[self.FEATURE_COLUMN_ORDER]

        # Retorna ambos DataFrames para uso en la siguiente etapa
        return df_items, df_features

    def load(self, df_items: pd.DataFrame, df_features: pd.DataFrame,
            output_dir: Path) -> None:
        # ETAPA 3 DEL PIPELINE: Exporta los DataFrames a archivos CSV
        # Define las rutas de salida para los dos archivos de resultado
        items_path = output_dir / 'kp_instances_unified.csv'
        features_path = output_dir / 'kp_features.csv'

        # Exporta DataFrame de elementos a CSV
        # index=False evita guardar índices de fila innecesarios
        df_items.to_csv(items_path, index=False)
        
        # Exporta DataFrame de características a CSV
        df_features.to_csv(features_path, index=False)

        # Imprime confirmación de exportación con información sobre cantidad de datos
        print(f"Saved {len(df_items)} item rows -> {items_path}")
        print(f"Saved {len(df_features)} instance features -> {features_path}")

    def run(self, output_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
        # ORQUESTA LA CADENA COMPLETA DE ETL (Extract → Transform → Load)
        # Este método ejecuta secuencialmente las tres etapas del pipeline
        
        # Etapa 1: Extrae instancias desde archivos en disco
        self.extract()
        
        # Etapa 2: Transforma instancias en DataFrames estructurados
        df_items, df_features = self.transform()
        
        # Etapa 3: Carga (exporta) los DataFrames a archivos CSV
        self.load(df_items, df_features, output_dir)
        
        # Retorna los DataFrames para inspección posterior o análisis adicional
        return df_items, df_features

## 3. Execute Pipeline

In [8]:
# Crea una instancia del pipeline y ejecuta la cadena completa de ETL
pipeline = KPPipeline(BASE_DIR)
df_items, df_features = pipeline.run(OUTPUT_DIR)

Found 16 subfolders: ['Set-100-128', 'Set-100-256', 'Set-100-32', 'Set-100-64', 'Set-200-128', 'Set-200-256', 'Set-200-32', 'Set-200-64', 'Set-25-128', 'Set-25-256', 'Set-25-32', 'Set-25-64', 'Set-50-128', 'Set-50-256', 'Set-50-32', 'Set-50-64']
  Set-100-128: 400 instances loaded
  Set-100-256: 400 instances loaded
  Set-100-32: 400 instances loaded
  Set-100-64: 400 instances loaded
  Set-200-128: 400 instances loaded
  Set-200-256: 400 instances loaded
  Set-200-32: 400 instances loaded
  Set-200-64: 400 instances loaded
  Set-25-128: 400 instances loaded
  Set-25-256: 400 instances loaded
  Set-25-32: 400 instances loaded
  Set-25-64: 400 instances loaded
  Set-50-128: 400 instances loaded
  Set-50-256: 400 instances loaded
  Set-50-32: 400 instances loaded
  Set-50-64: 400 instances loaded

Total: 6400 instances, 0 errors
Saved 600000 item rows -> outputs/kp_instances_unified.csv
Saved 6400 instance features -> outputs/kp_features.csv


## 4. Data Inspection

In [9]:
# Imprime información resumida del DataFrame de items (nivel de elemento)
print("Item-level dataset")
# Muestra las dimensiones del DataFrame (filas, columnas)
print(f"  Shape: {df_items.shape}")
# Lista los nombres de todas las columnas
print(f"  Columns: {list(df_items.columns)}")
# Muestra los valores únicos de heurísticas presentes en los datos
print(f"  Heuristics: {df_items['dominant_heuristic'].unique()}")
# Muestra los tipos de instancia únicos (easy/hard)
print(f"  Instance types: {df_items['instance_type'].unique()}")
# Muestra las primeras 10 filas del DataFrame
df_items.head(10)

Item-level dataset
  Shape: (600000, 6)
  Columns: ['filename', 'set', 'dominant_heuristic', 'instance_type', 'weight', 'profit']
  Heuristics: ['DEF' 'MAXPW' 'MAXP' 'MINW']
  Instance types: ['easy']


,filename,set,dominant_heuristic,instance_type,weight,profit
0,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,15,84
1,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,23,99
2,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,11,90
3,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,8,89
4,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,5,96
5,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,10,73
6,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,13,86
7,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,53,95
8,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,17,82
9,DEF_EASY_100_000.kp,Set-100-128,DEF,easy,19,100


In [10]:
# Imprime información resumida del DataFrame de características (nivel de instancia)
print("Feature-level dataset")
# Muestra las dimensiones del DataFrame (filas, columnas)
print(f"  Shape: {df_features.shape}")
# Lista los nombres de todas las columnas
print(f"  Columns: {list(df_features.columns)}")
# Muestra el rango de valores de capacity (capacidad mínima y máxima)
print(f"  Capacity range: [{df_features['capacity'].min()}, {df_features['capacity'].max()}]")
# Muestra el rango de valores de n_items (número mínimo y máximo de items)
print(f"  n_items range: [{df_features['n_items'].min()}, {df_features['n_items'].max()}]")
# Muestra estadísticas descriptivas (media, mediana, desv. estándar, etc.) de todas las columnas
df_features.describe()

Feature-level dataset
  Shape: (6400, 19)
  Columns: ['filename', 'set', 'dominant_heuristic', 'instance_type', 'n_items', 'capacity', 'weight_min', 'weight_max', 'profit_min', 'profit_max', 'w_mean', 'w_median', 'w_std', 'p_mean', 'p_median', 'p_std', 'wp_corr_raw', 'wp_corr', 'capacity_ratio']
  Capacity range: [32, 256]
  n_items range: [25, 200]


,n_items,capacity,weight_min,weight_max,profit_min,profit_max,w_mean,w_median,w_std,p_mean,p_median,p_std,wp_corr_raw,wp_corr,capacity_ratio
count,6400.00000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000,6400.000000
mean,93.75000,120.000000,1.543438,59.649219,2.314844,99.767031,0.555015,0.554434,0.278694,0.526943,0.545670,0.318574,-0.043144,0.478426,0.071550
std,67.02902,85.797146,1.366096,42.556434,2.828830,0.951853,0.095961,0.154013,0.026926,0.054101,0.116585,0.050698,0.159003,0.079506,0.057465
min,25.00000,32.000000,1.000000,14.000000,1.000000,89.000000,0.271000,0.181000,0.193000,0.343000,0.210000,0.172000,-0.607000,0.197000,0.015000
25%,43.75000,56.000000,1.000000,25.750000,1.000000,100.000000,0.492000,0.469000,0.258000,0.490000,0.465000,0.287000,-0.146250,0.427000,0.027000
50%,75.00000,96.000000,1.000000,44.000000,1.000000,100.000000,0.579000,0.594000,0.279000,0.519000,0.530000,0.304000,-0.037000,0.481000,0.054500
75%,125.00000,160.000000,2.000000,74.500000,3.000000,100.000000,0.627000,0.660000,0.298000,0.557000,0.600000,0.337000,0.066000,0.533000,0.114000
max,200.00000,256.000000,23.000000,128.000000,37.000000,100.000000,0.768000,0.875000,0.379000,0.768000,0.990000,0.485000,0.582000,0.791000,0.296000


In [11]:
# Lista de características a verificar (todas las características normalizadas y derivadas)
feature_columns = [
    'w_mean', 'w_median', 'w_std',
    'p_mean', 'p_median', 'p_std',
    'wp_corr_raw', 'wp_corr',
    'capacity_ratio',
]

# Cuenta el número de valores NaN (faltantes/no definidos) en cada característica
nan_count = df_features[feature_columns].isna().sum()

# Si hay al menos una característica con valores NaN, imprime los conteos
if nan_count.any():
    print("NaN counts per feature:")
    print(nan_count[nan_count > 0])  # Solo muestra características que tienen NaN
else:
    # Si no hay valores NaN en ninguna característica, confirma que los datos están completos
    print("No NaN values in any feature column.")

No NaN values in any feature column.


## 5. Feature Verification Against Paper Example

Section 2.2 (p. 12713) of Plata-González et al. (2019) provides a worked example:

> "Consider a subset of remaining items whose weights are (2, 2, 3, 4) and whose profits are (10, 5, 6, 15), respectively. Thus, each of the aforementioned features (in order) will take the values of (0.69, 0.63, 0.24, 0.60, 0.53, 0.30, 0.69)."

### Analysis of the first six features ($\bar{w}, \tilde{w}, \sigma_w, \bar{p}, \tilde{p}, \sigma_p$)

These six values are reproducible with `ddof=1` (sample standard deviation) when rounded to 2 decimal places. Using `ddof=0` (population standard deviation) yields $\sigma_w = 0.21$ and $\sigma_p = 0.26$, which do not match the paper's 0.24 and 0.30.

### Analysis of the seventh feature ($r = 0.69$)

The paper defines $r$ as: "weight-profit correlation, divided by two and shifted upwards by 0.5." For the given data:

| Metric | Raw value | After transformation ($\cdot / 2 + 0.5$) |
|--------|-----------|-------------------------------------------|
| Pearson $r$ | 0.689 | 0.845 |
| Spearman $\rho$ | 0.632 | 0.816 |

The paper's reported value of 0.69 matches the raw Pearson correlation without applying the transformation.

### Resolution

Both quantities are now stored explicitly:

| Column | Formula | Range | Interpretation |
|--------|---------|-------|----------------|
| `wp_corr_raw` | $r = \text{corr}(w, p)$ | $[-1, 1]$ | Raw Pearson; reproduces the paper's worked example value of 0.69 |
| `wp_corr` | $r / 2 + 0.5$ | $[0, 1]$ | Normalized per the paper's formula; consistent with the $[0,1]$ bound of all other features |

### Additional feature: `capacity_ratio`

$$\text{capacity\_ratio} = \frac{C}{\sum_{j=1}^{n} w_j}$$

This ratio measures the relative tightness of the knapsack constraint. A value close to 0 indicates a highly constrained instance; a value $\geq 1$ means the capacity is non-binding (all items can be packed). This feature is not part of Plata-González et al. (2019) but is a standard descriptor in the KP literature for characterizing instance difficulty (Pisinger, 2005).


In [12]:
# Reproduce el ejemplo trabajado de la Sección 2.2 (p. 12713) del artículo
test_instance = KPInstance(
    filename='verification_example',
    set_name='test',
    n_items=4,
    capacity=10,
    weights=np.array([2, 2, 3, 4], dtype=float),
    profits=np.array([10, 5, 6, 15], dtype=float),
    dominant_heuristic='test',
    instance_type='test',
)

# Calcula las características para la instancia de prueba
result = KPFeatureExtractor.compute(test_instance)

# Valores esperados del artículo (redondeados a 2 decimales)
paper_values = {
    'w_mean':   (0.69, ''),
    'w_median': (0.63, ''),
    'w_std':    (0.24, ''),
    'p_mean':   (0.60, ''),
    'p_median': (0.53, ''),
    'p_std':    (0.30, ''),
    'wp_corr_raw': (0.69, 'r de Pearson bruta -- coincide directamente con el ejemplo del artículo'),
    'wp_corr':     (0.845, 'r/2+0.5 -- fórmula del artículo; inconsistencia del ejemplo confirmada'),
    'capacity_ratio': (None, 'C / sum(w) = 10/11 ~ 0.909; no está en el artículo'),
}

# Imprime encabezado de tabla con características, valores calculados y esperados
print(f"{'Feature':<16} {'Computed':>10} {'Paper':>10} {'Match':>6}  Notes")
print('-' * 80)
# Itera sobre cada característica, compara valores calculados con esperados
for key, (expected, note) in paper_values.items():
    computed = result[key]
    if expected is None:
        match_str = 'N/A'
    else:
        # Determina si el valor calculado coincide con el esperado (tolerancia < 0.015)
        match_str = 'Y' if abs(computed - expected) < 0.015 else 'N'
    expected_str = f'{expected:.3f}' if expected is not None else '  ---'
    print(f"{key:<16} {computed:>10.3f} {expected_str:>10} {match_str:>6}  {note}")

# Muestra descriptores de rango sin normalizar (mínimo y máximo de pesos y ganancias)
print(f"\nRaw range descriptors: weight=[{result['weight_min']}, {result['weight_max']}], "
      f"profit=[{result['profit_min']}, {result['profit_max']}]")
# Muestra explicación de la reconciliación de correlación entre dos formatos
print(f"\nCorrelation reconciliation:")
print(f"  wp_corr_raw = {result['wp_corr_raw']:.3f}  -> coincide con el valor reportado 0.69 del artículo (r de Pearson bruta)")
print(f"  wp_corr     = {result['wp_corr']:.3f}  -> aplicación correcta de la fórmula del artículo (r/2+0.5)")
print(f"  Both are now stored. ")

Feature            Computed      Paper  Match  Notes
--------------------------------------------------------------------------------
w_mean                0.688      0.690      Y  
w_median              0.625      0.630      Y  
w_std                 0.239      0.240      Y  
p_mean                0.600      0.600      Y  
p_median              0.533      0.530      Y  
p_std                 0.303      0.300      Y  
wp_corr_raw           0.689      0.690      Y  r de Pearson bruta -- coincide directamente con el ejemplo del artículo
wp_corr               0.845      0.845      Y  r/2+0.5 -- fórmula del artículo; inconsistencia del ejemplo confirmada
capacity_ratio        0.909        ---    N/A  C / sum(w) = 10/11 ~ 0.909; no está en el artículo

Raw range descriptors: weight=[2, 4], profit=[5, 15]

Correlation reconciliation:
  wp_corr_raw = 0.689  -> coincide con el valor reportado 0.69 del artículo (r de Pearson bruta)
  wp_corr     = 0.845  -> aplicación correcta de la fórmula del